# Example Lecture -- AI-Generated Course Summary

[Open this notebook in Google Colab](https://colab.research.google.com/github/daanmeerburg/Statistics_meerburg_2026/blob/academic-staff-meeting-demo/Academic_Staff_Meeting/AI_Demo_Summary_Notebook.ipynb)

**Lecturer:** P. D. Meerburg

*This notebook is a demonstration for the academic staff meeting. It was generated from the structure and style of the 12 lecture notebooks in this course, and is meant to show how AI can create new course content that students can run and edit.*

##### Reading:

- Ivezic et al., *Statistics, Data Mining, and Machine Learning in Astronomy*, Chapters 3--5.
- David Hogg, *Data analysis recipes: Probability calculus for inference*.
- The course notebooks `Lectures/Lecture_01_PDM.ipynb` through `Lectures/Lecture_12_PDM.ipynb`.

---

>## What this example lecture is for <a class="anchor" id="one"></a>

This is not a replacement for the full lectures. It is a compact, executable overview that students can use after the course to connect the main ideas:

- probability distributions and uncertainty propagation,
- likelihoods and maximum-likelihood estimation,
- bootstrap uncertainty estimates,
- Bayesian posteriors and simple MCMC sampling,
- model comparison through evidence and Bayes factors.

The code examples are deliberately short. Students should not only run them, but also modify assumptions and inspect how the conclusions change.

>## Python setup <a class="anchor" id="setup"></a>

Run the next cell before running the examples. It follows the same idea as the lecture notebooks: use standard scientific Python tools, make plots readable, and install only when needed in Google Colab.

In [ ]:
# Execute this cell
import sys
import numpy as np
import matplotlib.pyplot as plt

# Install required libraries only when running on Google Colab.
if 'google.colab' in sys.modules:
    print("Running in Google Colab -- installing required packages...")
    !pip install scipy
else:
    print("Running locally -- assuming required packages are installed.")

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "text.usetex": False,
    "figure.figsize": (6, 4),
    "figure.dpi": 100,
})

try:
    get_ipython().run_line_magic('config', "InlineBackend.figure_format = 'retina'")
except NameError:
    pass

## Lecture Map

| Lecture | Topic | Main point |
|---|---|---|
| 1 | Introduction | Course goals, reproducible notebooks, GitHub/Colab workflow, and why statistics matters in modern physics. |
| 2 | Probability and statistics I | Events, conditional probability, Bayes' theorem, expectation values, variance, covariance, and the logic of probability models. |
| 3 | Probability and statistics II | Common distributions: uniform, Gaussian, log-normal, chi-square, Poisson, and Student's t. |
| 4 | Probability and statistics III | Multivariate Gaussian intuition, confidence regions, correlations, and resampling/up-sampling ideas. |
| 5 | Frequentist inference I | Likelihood functions, maximum-likelihood estimation, estimator uncertainty, and Gaussian approximations to the likelihood. |
| 6 | Frequentist inference II | Robust likelihoods, outliers, model comparison, bootstrap, jackknife, and a particle-physics invariant-mass dataset. |
| 7 | Frequentist inference III | Hypothesis testing, signal detection, p-values, the KS statistic, and parametric versus non-parametric tests. |
| 8 | Bayesian inference I | Bayes' theorem for inference, priors, conjugacy, maximum entropy, hierarchical Bayes, and physics examples. |
| 9 | Bayesian inference II | Bayesian model comparison, evidence, Bayes factors, AIC/BIC, Gaussian-vs-Cauchy examples, and coin-flip models. |
| 10 | Bayesian inference III | Monte Carlo ideas, Markov chains, detailed balance, stationarity, Metropolis sampling, and practical MCMC challenges. |
| 11 | Bayesian inference IV | MCMC diagnostics, adaptive proposals, HMC, emcee, PyMC, CosmoMC/Cobaya, and practical chain checking. |
| 12 | Bayesian inference V | Savage-Dickey ratios, product-space sampling, thermodynamic integration, nested sampling, dynesty, PolyChord, and UltraNest. |

## Core equations

### Probability rules

Sum rule:

$$
p(A \cup B)=p(A)+p(B)-p(A\cap B).
$$

Product rule:

$$
p(A\cap B)=p(A\mid B)p(B)=p(B\mid A)p(A).
$$

Bayes' theorem:

$$
p(\theta\mid D,M)=\frac{p(D\mid\theta,M)p(\theta\mid M)}{p(D\mid M)}.
$$

### Expectation and variance

$$
\mathbb{E}[X]=\int x p(x)\,dx, \qquad
\mathrm{Var}(X)=\mathbb{E}[(X-\mathbb{E}[X])^2].
$$

For independent measurements with variance $\sigma^2$,

$$
\mathrm{Var}(\bar X)=\frac{\sigma^2}{N}.
$$

## Frequentist inference in one page

The likelihood function treats the observed data as fixed and the parameter as variable:

$$
L(\theta)=p(D\mid \theta).
$$

The maximum-likelihood estimator is

$$
\hat\theta_{\rm MLE}=\arg\max_\theta L(\theta)
=\arg\max_\theta \log L(\theta).
$$

Near the maximum, many likelihoods can be approximated as Gaussian:

$$
\log L(\theta)\simeq \log L(\hat\theta)-\frac{1}{2}
\frac{(\theta-\hat\theta)^2}{\sigma_\theta^2}.
$$

Hypothesis tests compare a statistic to its null distribution. A p-value is

$$
p = P(T \ge T_{\rm obs}\mid H_0)
$$

for an upper-tail test. It is not the probability that the null hypothesis is true.

## Bayesian inference in one page

Bayesian inference combines prior information and the likelihood:

$$
p(\theta\mid D,M)=\frac{p(D\mid\theta,M)p(\theta\mid M)}{\mathcal Z},
\qquad
\mathcal Z=p(D\mid M).
$$

The evidence is the prior-averaged likelihood:

$$
\mathcal Z=\int p(D\mid\theta,M)p(\theta\mid M)\,d\theta.
$$

It is used for model comparison through Bayes factors:

$$
B_{10}=\frac{\mathcal Z_1}{\mathcal Z_0}.
$$

Markov-chain Monte Carlo generates dependent samples from the posterior. For a symmetric proposal, the Metropolis acceptance probability is

$$
\alpha=\min\left(1,\frac{p(\theta'\mid D)}{p(\theta\mid D)}\right).
$$

Practical checks include trace plots, autocorrelation, effective sample size, and comparisons between independent chains.

>## Coding example 1: distributions, CLT, and bootstrap <a class="anchor" id="code-one"></a>

This example connects Lectures 2--4 and 6. We start from a deliberately non-Gaussian parent distribution, the exponential distribution,

$$
p(x\mid \lambda)=\lambda e^{-\lambda x}, \qquad x\ge 0,
$$

and draw many independent datasets of size $N$. For each dataset the code computes the sample mean

$$
\bar x = \frac{1}{N}\sum_{i=1}^{N}x_i.
$$

The first panel shows the sampling distribution of $\bar x$. Even though the original data are skewed, the distribution of the mean becomes more Gaussian as $N$ grows. This is the central-limit-theorem idea in code.

The second panel takes one observed dataset and estimates the uncertainty on its mean by bootstrap resampling. Each bootstrap sample is drawn with replacement from the observed data, and the mean is recomputed. The spread of those bootstrap means approximates the uncertainty on $\bar x$.

**What students can change.** Try changing `n`, `n_experiments`, the exponential `scale`, or replace `rng.exponential` by `rng.normal`, `rng.lognormal`, or `rng.poisson`. You can also replace `.mean()` by `.median()` and see why the bootstrap is useful for statistics whose uncertainty is not as simple analytically.

**Take-away message.** The distribution of raw measurements and the distribution of an estimator are not the same object. The central limit theorem explains why means often look Gaussian; the bootstrap gives an empirical way to estimate estimator uncertainty when formulas are inconvenient.

In [ ]:
rng = np.random.default_rng(42)
n = 25
n_experiments = 5000

# Draw exponential data, then look at the distribution of sample means.
samples = rng.exponential(scale=1.0, size=(n_experiments, n))
means = samples.mean(axis=1)

# Bootstrap the mean of one observed dataset.
observed = rng.exponential(scale=1.0, size=n)
boot_means = np.array([
    rng.choice(observed, size=n, replace=True).mean()
    for _ in range(5000)
])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(means, bins=40, density=True, alpha=0.75)
ax[0].set_title("Sampling distribution of sample means")
ax[0].set_xlabel(r"$\bar x$")

ax[1].hist(boot_means, bins=40, density=True, alpha=0.75, color="tab:orange")
ax[1].axvline(observed.mean(), color="k", ls="--", label="observed mean")
ax[1].set_title("Bootstrap distribution of the mean")
ax[1].set_xlabel(r"bootstrap $\bar x$")
ax[1].legend()
plt.tight_layout()

>## Coding example 2: maximum likelihood for a straight-line model <a class="anchor" id="code-two"></a>

This example connects Lectures 5--7. We generate data from a straight-line model with Gaussian measurement noise,

$$
y_i = a + b x_i + \epsilon_i, \qquad \epsilon_i\sim\mathcal N(0,\sigma^2).
$$

For known $\sigma$, the likelihood is

$$
L(a,b)=\prod_i \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left[-\frac{(y_i-a-bx_i)^2}{2\sigma^2}\right].
$$

The code minimizes the negative log-likelihood,

$$
-\log L(a,b)=\frac{1}{2}\sum_i\left[\frac{(y_i-a-bx_i)^2}{\sigma^2}+\log(2\pi\sigma^2)\right],
$$

and plots the maximum-likelihood line on top of the noisy data.

**What students can change.** Try increasing `sigma`, reducing the number of data points, changing the true intercept/slope, or adding one outlier by hand. Then ask: does the best-fit line move? Would a Gaussian likelihood still be a sensible model? This naturally leads to robust likelihoods and outlier models.

**Take-away message.** Maximum likelihood is not just a fitting recipe. It is a statement about the data-generating model. Changing the assumed noise model changes what "best fit" means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(7)
x = np.linspace(0, 10, 30)
sigma = 0.7
y = 1.5 + 0.8 * x + rng.normal(0, sigma, size=len(x))

def neg_log_likelihood(params):
    a, b = params
    model = a + b * x
    return 0.5 * np.sum(((y - model) / sigma)**2 + np.log(2 * np.pi * sigma**2))

result = minimize(neg_log_likelihood, x0=[0.0, 1.0])
a_hat, b_hat = result.x
print(f"MLE intercept = {a_hat:.3f}, slope = {b_hat:.3f}")

plt.errorbar(x, y, yerr=sigma, fmt="o", label="data")
plt.plot(x, a_hat + b_hat * x, label="MLE fit")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

>## Coding example 3: Bayesian posterior and a simple Metropolis sampler <a class="anchor" id="code-three"></a>

This example connects Lectures 8--12. We infer the unknown mean $\mu$ of Gaussian data with known standard deviation $\sigma$ and a Gaussian prior,

$$
p(\mu)\propto \exp\left[-\frac{\mu^2}{2\tau^2}\right].
$$

The posterior is proportional to likelihood times prior:

$$
p(\mu\mid D)\propto
\exp\left[-\frac{1}{2}\sum_i\frac{(x_i-\mu)^2}{\sigma^2}\right]
\exp\left[-\frac{\mu^2}{2\tau^2}\right].
$$

The code samples this posterior with a simple Metropolis algorithm. A proposed step $\mu'$ is accepted with probability

$$
\alpha=\min\left(1,\frac{p(\mu'\mid D)}{p(\mu\mid D)}\right).
$$

The trace plot shows how the chain moves through parameter space; the histogram after burn-in approximates the posterior density.

**What students can change.** Try changing `proposal_width`: too small gives slow exploration, too large gives many rejected proposals. Try changing `tau` to make the prior stronger or weaker. Try changing the number of data points and watch the posterior become narrower as the data dominate.

**Take-away message.** Bayesian inference returns a distribution, not only a point estimate. MCMC is a practical way to represent that distribution by samples, but those samples must be checked for mixing and convergence.

In [ ]:
rng = np.random.default_rng(123)
data = rng.normal(loc=0.6, scale=1.0, size=40)
sigma = 1.0
tau = 2.0

def log_posterior(mu):
    log_like = -0.5 * np.sum((data - mu)**2 / sigma**2)
    log_prior = -0.5 * mu**2 / tau**2
    return log_like + log_prior

n_steps = 12000
chain = np.empty(n_steps)
chain[0] = 0.0
proposal_width = 0.35

for i in range(1, n_steps):
    proposal = chain[i - 1] + rng.normal(0, proposal_width)
    log_alpha = log_posterior(proposal) - log_posterior(chain[i - 1])
    if np.log(rng.uniform()) < log_alpha:
        chain[i] = proposal
    else:
        chain[i] = chain[i - 1]

burn = 2000
posterior_samples = chain[burn:]
print(f"posterior mean = {posterior_samples.mean():.3f}")
print(f"posterior std  = {posterior_samples.std(ddof=1):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(chain, lw=0.5)
ax[0].axvline(burn, color="r", ls="--", label="burn-in")
ax[0].set_title("Trace plot")
ax[0].set_xlabel("step")
ax[0].set_ylabel(r"$\mu$")
ax[0].legend()

ax[1].hist(posterior_samples, bins=40, density=True, alpha=0.75)
ax[1].set_title("Posterior samples")
ax[1].set_xlabel(r"$\mu$")
plt.tight_layout()

## How to use this as a student study notebook

A useful workflow is:

1. Read the lecture map and identify weak topics.
2. Re-run each coding example and change one modeling assumption.
3. Connect each equation to one physical example from the lectures.
4. For exam preparation, practice explaining the difference between likelihood, posterior, evidence, p-value, confidence interval, and credible interval in words.